# CNNs — Convolutional Neural Networks

CNNs replace fully-connected layers with **convolutional** layers that learn spatial filters.  
Instead of connecting every pixel to every neuron, each filter slides over the image — far fewer parameters, translation-invariant features.

This notebook: classify MNIST digits with a small CNN.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Load MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

train_ds = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64)

print(f"Train: {len(train_ds)}  Test: {len(test_ds)}")
print(f"Image shape: {train_ds[0][0].shape}")  # (1, 28, 28) — channels, height, width

In [ ]:
# Visualize a few samples
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(imgs[i].squeeze(), cmap='gray')
    ax.set_title(labels[i].item())
    ax.axis('off')
plt.show()

## CNN Architecture

```
Input (1×28×28)
  → Conv2d(1→32, 3×3) + ReLU   → (32×26×26)
  → MaxPool2d(2×2)              → (32×13×13)
  → Conv2d(32→64, 3×3) + ReLU  → (64×11×11)
  → MaxPool2d(2×2)              → (64×5×5)
  → Flatten                    → (1600,)
  → Linear(1600→128) + ReLU
  → Linear(128→10)             → logits for 10 classes
```

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)   # 1 input channel (grayscale)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 5 * 5, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # (32, 13, 13)
        x = self.pool(F.relu(self.conv2(x)))  # (64, 5, 5)
        x = x.flatten(start_dim=1)            # (1600,)
        x = F.relu(self.fc1(x))
        return self.fc2(x)                    # raw logits


model = CNN().to(DEVICE)
print(model)

# Count parameters
total = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total:,}")

## Understand Conv2d

`nn.Conv2d(in_channels, out_channels, kernel_size)` learns `out_channels` filters of shape `(in_channels × kH × kW)`.  
Each filter slides over the input producing a 2D feature map (activation map).

In [ ]:
# Single conv layer demo
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3)
x = torch.randn(1, 1, 28, 28)   # batch=1, channels=1, H=28, W=28
out = conv(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")    # (1, 8, 26, 26) — 8 filters, spatial shrinks by 2

In [ ]:
# padding='same' keeps spatial size unchanged
conv_same = nn.Conv2d(1, 8, kernel_size=3, padding=1)
out_same = conv_same(x)
print(f"With padding=1: {out_same.shape}")  # (1, 8, 28, 28)

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total   += yb.size(0)

    print(f"Epoch {epoch+1}  loss={total_loss/len(train_loader):.4f}  test_acc={correct/total:.4f}")

## Visualize Predictions

In [ ]:
model.eval()
imgs, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(imgs.to(DEVICE)).argmax(1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(imgs[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(f'true: {labels[i].item()}')
    axes[1, i].imshow(imgs[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    color = 'green' if preds[i] == labels[i] else 'red'
    axes[1, i].set_title(f'pred: {preds[i].item()}', color=color)
plt.tight_layout()
plt.show()

## What to try next

- Add `nn.BatchNorm2d` after each conv layer — faster, more stable training
- Add `nn.Dropout2d` before the classifier — regularization
- Try on CIFAR-10 (colour images, 10 classes) — needs 3 input channels
- Use a pretrained backbone (ResNet, VGG) from `torchvision.models` — transfer learning